In [1]:
# This cell is removed with the tag: "remove-input"
# As such, it will not be shown in documentation

import warnings
warnings.filterwarnings('ignore')


(Tutorial_Get_bonded_atom_pairs)=
# Get bonded atom pairs

*Looking up standard intra-group covalent bond index pairs.*

The function {func}`molsysmt.element.group.get_bonded_atom_pairs` returns standard chemical bond pairs given a group name and a list of present atom names.

:::{versionadded} 1.0.0
:::

:::{admonition} API documentation
:class: dropdown

Follow this link for a detailed description of the input arguments, raised errors, and returned objects of this function: {func}`molsysmt.element.group.get_bonded_atom_pairs`.
:::


## Basic usage

Let's retrieve the covalent bond topology for an alanine residue:


In [2]:
import molsysmt as msm


In [3]:
atom_names = ['N', 'CA', 'C', 'O', 'CB', 'H', 'HA', 'HB1', 'HB2', 'HB3']
bonds = msm.element.group.get_bonded_atom_pairs('ALA', atom_names)
print('Alanine bonded atom pairs (index pairs):', bonds)


Alanine bonded atom pairs (index pairs): [[0, 1], [0, 5], [1, 2], [1, 4], [1, 6], [2, 3], [4, 7], [4, 8], [4, 9]]


## Scoping reference names

The keyword-only `naming_scheme` selects a group vocabulary. Ambiguous names
require context; verified Amber state layouts cannot be selected by CCD or legacy
labels. Only supported exact-name layouts are mapped, without parent substitution.
Standard labels retain supported terminal layouts; internal histidine layouts
are restricted to the checked HID/HIE/HIP references matching observed atoms.
Unsupported layouts remain unassessed. No chemical state is assigned and the
original names/index map remain unchanged. Optional Ackredit records reached data.
See {ref}`user-foundations-group-naming-schemes`.


In [4]:
print(msm.element.group.get_bonded_atom_pairs(
    'GLH', ['CD', 'OE2', 'HE2'], atom_indices=[70, 80, 90],
    naming_scheme='amber'))

[[np.int64(70), np.int64(80)], [np.int64(80), np.int64(90)]]


## Preserving names and index maps

The saccharide, small-molecule and terminal-capping helpers accept the same
one-dimensional atom-name sequence as a list, tuple or NumPy array. Read-only
and noncontiguous arrays work too. Empty names give an empty list. Nested names,
scalar names and non-string entries raise `ArgumentError` for recognized groups.
These three reference helpers retain the first occurrence of a repeated name;
they do not choose alternate locations.

Their outputs are lists of two-index lists, rather than atom-name pairs. With
`atom_indices=None`, endpoints refer to positions in your supplied name sequence.
With explicit `atom_indices`, supply one integer per name in the same order;
reordering that sequence changes the name-to-index map. For example, the NAG
names C1/C2/C3 mapped to 20/10/30 give `[[10, 20], [10, 30]]`. These are indices,
not atom IDs, and no atoms are selected from a molecular system.

For these three helpers each pair puts its smaller endpoint first. `sorted=True`
sorts the rows lexicographically; `False` keeps template bond order. Unknown
small-molecule names or unmatched layouts give `[]`; the saccharide and capping
helpers retain `InternalAlgorithmError` for those cases. Names and index inputs
are never modified.

See the specific APIs for
{func}`molsysmt.element.group.saccharide.get_bonded_atom_pairs`,
{func}`molsysmt.element.group.small_molecule.get_bonded_atom_pairs` and
{func}`molsysmt.element.group.terminal_capping.get_bonded_atom_pairs`.
This is a reference-catalogue lookup; it does not read coordinates, assign a
chemical state, add bonds, or replace the stored graph of an actual system.

## Comparing family conventions

All seven public bond-pair entry points use integer indices for valid input
maps. An omitted `atom_indices` map means positions in the supplied names;
an explicit map must contain one integer for every name, in matching order.
These indices are unrelated to atom IDs. Use the family helper's contract for
unknown groups and unmatched layouts:

| Route | Successful pairs and ordering | Empty or missing reference result |
| --- | --- | --- |
| Amino acid | Smaller endpoint first; repeated names expand matching reference edges over all supplied occurrences. | Known empty names give `[]`; unknown groups and unmatched layouts give `None`. |
| Ion | Smaller endpoint first; repeated names use their first occurrence. | One name gives `[]` before catalogue lookup; known empty names give `[]`; unmatched multi-atom layouts give `None`; other unknown-group queries raise `InternalAlgorithmError`. |
| Saccharide | Smaller endpoint first; first repeated-name occurrence. | Known empty names give `[]`; unknown groups or unmatched layouts raise `InternalAlgorithmError`. |
| Small molecule | Smaller endpoint first; first repeated-name occurrence. | Empty names, unknown groups and unmatched layouts give `[]`. |
| Terminal cap | Smaller endpoint first; first repeated-name occurrence. | Known empty names give `[]`; unknown groups or unmatched layouts raise `InternalAlgorithmError`. |
| Water | Oxygen first; first two hydrogens in supplied name order. | Fewer than three names give `[]`; complete queries must describe a water layout. |
| General dispatcher | Uses the selected family's result and ordering. | Unsupported lipid/nucleotide/polysaccharide routes and unknown families give `None`; family errors propagate. |

`sorted=True` orders pair rows lexicographically. `sorted=False` keeps template
bond order, or hydrogen encounter order for water. Sorting rows does not reverse
water endpoints: O/H1/H2 mapped to 1040/1009/1030 gives
`[[1040, 1009], [1040, 1030]]`, with the oxygen index first.

Reference lookups neither examine a stored molecular graph nor modify one.
For connectivity of an actual system, query
{func}`molsysmt.chemical_states.get_bondgraph` with its chemical state and atom
selection. All seven entry points validate explicit maps before reference lookup.
Supply one nonnegative int64-range integer for every name; booleans, strings,
floats and nested maps raise `ArgumentError`. A length mismatch raises
`ArgumentLengthError`, carrying expected and actual lengths. Repeated indices
are allowed: this correspondence is not a permutation. Empty names require
an empty map or `None`. Readonly and strided arrays remain supported; units and masks are rejected.

Amino-acid and ion helpers, including their dispatcher routes, retain scalar
integer maps for single-name queries. The other families require a flat map.
Their existing unknown/empty/layout outcomes apply after input validation.
Amino-acid and ion `skip_digestion=True` calls require already valid inputs;
no new skip flag is added to the other helpers.

:::{seealso}
:class: dropdown

**Related Tools & References**

- {ref}`Get group type <Tutorial_Get_group_type>`: Identify group category with {func}`molsysmt.element.group.get_group_type`.
- {ref}`Is group type <Tutorial_Is_group_type>`: Check specific group type with {func}`molsysmt.element.group.is_group_type`.
:::
